# Notebook for setup for Noe and Porter datasets.
This notebook prepares cleaned OOD60/Porter fold switcher datasets for \
likelihood calculation experiments and generates Slurm scripts to run \
the experiments. 

The notebook generates two directories of files for each dataset: one \
contains pairs prepped with PDBFixer, which are used for calculating \
likelihoods with Boltz. This directory will also contain the necessary \
.yaml file required by Boltz. The other contains pairs prepped with \
pdb4amber as well as .inpcrd and .prmtop files required for likelihood \
calculations with OMM. 

Requires:
- Raw OOD60 dataset (https://zenodo.org/records/15672282/files/bioemu-benchmarks_code_release.zip?download=1)
- Raw fold switcher dataset xlsx file (https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-024-51801-z/MediaObjects/41467_2024_51801_MOESM4_ESM.xlsx)
- AmberTools installed

Prepped datasets are saved in the following file structure: \
dataset/ \
&nbsp;&nbsp;&nbsp;&nbsp; pair_name/ \
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; {PDB ID}\_{Chain ID}.pdb \
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; {PDB ID}\_{Chain ID}.pdb \
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; {PDB ID}\_{Chain ID}.inpcrd (if present) \
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; {PDB ID}\_{Chain ID}.prmtop (if present)

In [ ]:
import os
import mdtraj as md
import yaml
import numpy as np
import pandas as pd
import subprocess
import tempfile
import shutil

from Bio import PDB
from Bio.PDB import PDBList, PDBParser, Select, PDBIO, PPBuilder
from Bio.PDB.Polypeptide import is_aa
from Bio.Data.IUPACData import protein_letters_3to1

from openmm.app import PDBFile
from pathlib import Path
from difflib import SequenceMatcher
from tqdm import tqdm

from pdbfixer.pdbfixer.pdbfixer import PDBFixer

# Change as appropriate.
FINAL_OOD_DIR = '/home/ethanz/data_boltz_likelihood/ood60_clean'
OOD_AMBER = '/home/ethanz/data_boltz_likelihood/ood60_amber'
RAW_DIR = '/home/ethanz/data_boltz_likelihood/ood60/reference'
os.makedirs(FINAL_OOD_DIR, exist_ok=True)
os.makedirs(OOD_AMBER, exist_ok=True)

FINAL_PORTER_DIR = '/home/ethanz/data_boltz_likelihood/porter'
PORTER_AMBER = '/home/ethanz/data_boltz_likelihood/porter_amber'
RAW_PORTER = '/home/ethanz/data_boltz_likelihood/porter.xlsx'
os.makedirs(FINAL_PORTER_DIR, exist_ok=True)
os.makedirs(PORTER_AMBER, exist_ok=True)

## Helper functions.

In [2]:
def extract_matching_chain(pdb_id, backbone_pdb_path, tolerance=0.1):
    """Extract a full-atom chain from a raw PDB given a PDB of a
    heavy-atom chain extracted from the raw PDB.

    Parameters
    ----------
    pdb_id : str
        PDB ID of the chain's original file. 

    backbone_pdb_path : str
        Path to the backbone-only PDB file (OOD60).

    
    Returns
    -------
    best_chain : Bio.PDB.Chain
    """
    # Load backbone-only structure
    parser = PDB.PDBParser(QUIET=True)
    backbone_struct = parser.get_structure("backbone", backbone_pdb_path)
    backbone_atoms = [a for a in backbone_struct.get_atoms() if a.get_name() in ("N", "CA", "C")]

    # Download full PDB
    pdbl = PDB.PDBList()
    full_pdb_file = pdbl.retrieve_pdb_file(pdb_id, file_format='pdb', pdir='.')
    full_struct = parser.get_structure(pdb_id, full_pdb_file)

    model = full_struct[0]

    min_rmsd = float('inf')
    best_chain = None

    for chain in model:
        # Extract backbone atoms from this chain
        chain_atoms = [a for a in chain.get_atoms() if a.get_name() in ("N", "CA", "C")]
        # Try to match length first
        if len(chain_atoms) != len(backbone_atoms):
            continue

        # Compute RMSD
        coords1 = np.array([a.get_coord() for a in backbone_atoms])
        coords2 = np.array([a.get_coord() for a in chain_atoms])

        diff = coords1 - coords2
        rmsd = np.sqrt(np.mean(np.sum(diff**2, axis=1)))

        if rmsd < min_rmsd:
            min_rmsd = rmsd
            best_chain = chain

    if min_rmsd > tolerance:
        raise ValueError(f"No matching chain found within RMSD tolerance (min_rmsd = {min_rmsd}).")

    os.remove(full_pdb_file)

    print(f"Returning matching chain (RMSD={min_rmsd:.3f}).")
    return best_chain

def crop_chain_by_sequence(chain, matching_subseq, output_pdb_path):
    """Save a cropped version of a Chain object based on a matching 
    sequence string.
    """
    # Step 1: Extract sequence from the chain
    residues = [res for res in chain if is_aa(res)]
    seq = ''.join(protein_letters_3to1[res.get_resname().capitalize()] for res in residues)

    # Step 2: Find matching subsequence
    idx = seq.find(matching_subseq)
    if idx == -1:
        print("Matching subsequence not found in chain sequence.")
        return -1

    # Step 3: Map positions back to residue objects
    matched_residues = residues[idx : idx + len(matching_subseq)]
    matched_ids = {res.id for res in matched_residues}

    # Step 4: Save only matched residues
    class SequenceSelect(Select):
        def accept_residue(self, residue):
            return residue.id in matched_ids

    # Wrap chain in dummy structure for saving
    from Bio.PDB.StructureBuilder import StructureBuilder
    builder = StructureBuilder()
    builder.init_structure("cropped")
    builder.init_model(0)
    builder.structure[0].add(chain)

    io = PDBIO()
    io.set_structure(builder.get_structure())
    io.save(output_pdb_path, SequenceSelect())

def longest_common_substring(s1, s2):
    matcher = SequenceMatcher(None, s1, s2, autojunk=False)
    match = matcher.find_longest_match(0, len(s1), 0, len(s2))
    if match.size == 0:
        return None  # No match
    return (match.a, match.a + match.size), (match.b, match.b + match.size)

def write_yaml(sequence_str, out_path):
    data = {
        "version": 1,
        "sequences": [
            {
                "protein": {
                    "id": "A",
                    "sequence": sequence_str
                }
            }
        ]
    }
    with open(out_path, "w") as f:
        yaml.dump(data, f, sort_keys=False)

## OOD60.

### Creating dataset and Amber files.

In [3]:
pdbl = PDBList()
parser = PDBParser(QUIET=True)

ppb = PPBuilder()
matched = False

'''The OOD60 dataset provides us with heavy-atom-only PDB files of pairs
of protein chains. To process this raw dataset, the following is done: 

1) Only considering pairs with large (>75%) continuous sequence overlap, 
fetch the original raw, all-atom PDBs and save the corresponding
all-atom chain (OOD60 chains are in their original positions, so just 
find the near-zero RMSD chain of the original PDB file). 

2) Crop the all-atom chain pair to the maximum sequence overlap region.

3) Use PDBFixer to fill in missing heavy atoms. 
'''
for subdir in os.listdir(RAW_DIR):
    pdbs = os.listdir(os.path.join(RAW_DIR, subdir))

    # Finding sequence overlap between OOD60 PDB pair.
    traj1 = md.load_pdb(os.path.join(RAW_DIR, subdir, pdbs[0]))
    traj2 = md.load_pdb(os.path.join(RAW_DIR, subdir, pdbs[1]))

    seq1 = traj1.topology.to_fasta()[0]
    seq2 = traj2.topology.to_fasta()[0]

    seq1idxs, seq2idxs = longest_common_substring(seq1, seq2)
    
    cut_seq1 = seq1[seq1idxs[0]:seq1idxs[1]]
    cut_seq2 = seq2[seq2idxs[0]:seq2idxs[1]]
    assert cut_seq1 == cut_seq2, 'Mismatched sequences!'

    if max(len(seq1), len(seq2)) * 0.75 < len(cut_seq1):
        tempdir = tempfile.mkdtemp()
        outdir = os.path.join(FINAL_OOD_DIR, subdir)
        outdir_amber = os.path.join(OOD_AMBER, subdir)
        os.makedirs(outdir, exist_ok=True)
        os.makedirs(outdir_amber, exist_ok=True)

        write_yaml(cut_seq1, os.path.join(outdir, 'seq.yaml'))

        pdb_id_1 = pdbs[0][:4]
        pdb_id_2 = pdbs[1][:4]
            
        # Extracting the proper chain from the original PDB. 
        all_atom_chain_1 = extract_matching_chain(
            pdb_id=pdb_id_1, 
            backbone_pdb_path=os.path.join(RAW_DIR, subdir, pdbs[0])        
        )
        all_atom_chain_2 = extract_matching_chain(
            pdb_id=pdb_id_2,
            backbone_pdb_path=os.path.join(RAW_DIR, subdir, pdbs[1]) 
        )

        outname1 = os.path.join(outdir, pdbs[0]) # eg. outdir/6AJM_C.pdb.
        outname2 = os.path.join(outdir, pdbs[1])

        # Cropping all-atom chains.
        crop_chain_by_sequence(
            chain=all_atom_chain_1,
            matching_subseq=cut_seq1,
            output_pdb_path=os.path.join(tempdir, 'temp1.pdb')
        )
        crop_chain_by_sequence(
            chain=all_atom_chain_2,
            matching_subseq=cut_seq1,
            output_pdb_path=os.path.join(tempdir, 'temp2.pdb')
        )

        '''Two prep steps occur: 1) using pdb4amber + tleap to create 
        the necessary Amber files for likelihood calculation 2) using 
        pdbfixer to create likelihood-ready structures. There are small
        differences and incompatibilities between pdb4amber and pdbfixer
        outputs, hence the necessity of using both.
        '''

        # Using pdb4amber to create tleap inputs.
        pdb4amber_cmd = [
            "pdb4amber",
            "-i", f"{os.path.join(tempdir, 'temp1.pdb')}",
            "-o", f"{os.path.join(tempdir, 'temp1p.pdb')}",
            "--nohyd",
        ]
        result = subprocess.run(
            pdb4amber_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        pdb4amber_cmd = [
            "pdb4amber",
            "-i", f"{os.path.join(tempdir, 'temp2.pdb')}",
            "-o", f"{os.path.join(tempdir, 'temp2p.pdb')}",
            "--nohyd",
        ]
        result = subprocess.run(
            pdb4amber_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        if result.returncode != 0:
            print("stderr:\n", result.stderr)

        # tleap-ing pdb4amber outputs.
        prmtop_savepath = os.path.join(outdir_amber, f'{pdbs[0][:6]}.prmtop')
        inpcrd_savepath = os.path.join(outdir_amber, f'{pdbs[0][:6]}.inpcrd')
        pdb_savepath = os.path.join(outdir_amber, pdbs[0])
        tleap_input = f"""
        source leaprc.ff99SBxildn
        mol = loadPDB "{os.path.join(tempdir, 'temp1p.pdb')}"
        saveAmberParm mol "{prmtop_savepath}" "{inpcrd_savepath}"
        savePDB mol "{pdb_savepath}"
        quit
        """
        result = subprocess.run(
            ["tleap", "-f", "-"],  # "-" means read from stdin
            input=tleap_input.encode(),
            check=True
        )
         # tleap-ing pdb4amber outputs.
        prmtop_savepath = os.path.join(outdir_amber, f'{pdbs[1][:6]}.prmtop')
        inpcrd_savepath = os.path.join(outdir_amber, f'{pdbs[1][:6]}.inpcrd')
        pdb_savepath = os.path.join(outdir_amber, pdbs[1])
        tleap_input = f"""
        source leaprc.ff99SBxildn
        mol = loadPDB "{os.path.join(tempdir, 'temp2p.pdb')}"
        saveAmberParm mol "{prmtop_savepath}" "{inpcrd_savepath}"
        savePDB mol "{pdb_savepath}
        quit
        """
        result = subprocess.run(
            ["tleap", "-f", "-"],  # "-" means read from stdin
            input=tleap_input.encode(),
            check=True
        )

        # Fixing using pdbfixer to get likelihood-ready PDBs.
        fixer1 = PDBFixer(filename=os.path.join(tempdir, 'temp1.pdb'))
        fixer1.findMissingResidues()
        fixer1.findMissingAtoms()
        fixer1.addMissingAtoms()
        PDBFile.writeFile(fixer1.topology, fixer1.positions, open(outname1, 'w'))

        fixer2 = PDBFixer(filename=os.path.join(tempdir, 'temp2.pdb'))
        fixer2.findMissingResidues()
        fixer2.findMissingAtoms()
        fixer2.addMissingAtoms()
        PDBFile.writeFile(fixer2.topology, fixer2.positions, open(outname2, 'w'))

        shutil.rmtree(tempdir)

Returning matching chain (RMSD=0.000).
Returning matching chain (RMSD=0.000).
-I: Adding /home/ethanz/ambertools25/dat/leap/prep to search path.
-I: Adding /home/ethanz/ambertools25/dat/leap/lib to search path.
-I: Adding /home/ethanz/ambertools25/dat/leap/parm to search path.
-I: Adding /home/ethanz/ambertools25/dat/leap/cmd to search path.
-f: Source -.

Welcome to LEaP!
(no leaprc in search path)
Sourcing: /home/ethanz/ambertools25/dat/leap/cmd/leaprc
----- Source: /home/ethanz/ambertools25/dat/leap/cmd/leaprc.ff99SBxildn
----- Source of /home/ethanz/ambertools25/dat/leap/cmd/leaprc.ff99SBxildn done
Log file: ./leap.log
Loading parameters: /home/ethanz/ambertools25/dat/leap/parm/parm99x.dat
Reading title:
PARM99 for DNA,RNA,AA, organic molecules, TIP3P wat. Polariz.& LP incl.02/04/99
Loading parameters: /home/ethanz/ambertools25/dat/leap/cmd/frcmod.ff99SBxildn
Reading force field modification type file (frcmod)
Reading title:
Modification/update of parm99.dat (Hornak & Simmerling) +

### Writing a Slurm/bash script to run likelihood experiment on OOD60.

In [ ]:
DATA_DIR = '/home/ethanz/data_boltz_likelihood/ood60_clean'

job_script_path = "run_OOD60_likelihoods.sh"
exp_names = sorted(os.listdir(DATA_DIR))
head_inits = [
    os.path.join(DATA_DIR, exp_name)
    for exp_name in exp_names
]
yamls = [
    os.path.join(head_init, 'seq.yaml')
    for head_init in head_inits
]

# Define multiple argument sets for each run
runs = [
    {
        "exp_name": f"{exp_names[i]}_likelihoods",
        "head_init": head_inits[i],
        "yaml": yamls[i]
    }
    for i in range(len(exp_names))
]

with open(job_script_path, "w") as f:
    f.write("#!/bin/bash\n")
    f.write("#SBATCH --gres=gpu:1\n")
    f.write("#SBATCH --job-name=OOD60_likelihoods\n")

    f.write("source .venv/bin/activate\n\n")

    for run in runs:
        f.write(f'echo "Starting {run["exp_name"]}"\n')

        f.write("boltz predict \\\n")
        f.write(f"    {run["yaml"]} \\\n")
        f.write("    --atol 0.000001 \\\n")
        f.write("    --rtol 0.001 \\\n")
        f.write("    --model boltz1 \\\n")
        f.write("    --out_dir ./predictions/ood60_final \\\n")
        f.write("    --use_msa_server \\\n")
        f.write("    --confidence False \\\n")
        f.write("    --output_format pdb \\\n")
        f.write("    --save_conditioning_args False \\\n")
        f.write("    --mode likelihood \\\n")
        f.write(f"    --experiment_name {run['exp_name']} \\\n")
        f.write(f"    --slurm_path {job_script_path} \\\n")
        f.write("    --likelihood_mode hutchinson \\\n")
        f.write("    --hutchinson_samples 20 \\\n") # ODE batch size is 1 by default.
        f.write(f'    --head_init {run["head_init"]} \\\n')

        f.write("\n")  # Separate commands with a blank line

print(f"Wrote job script to {job_script_path}")

Wrote job script to run_OOD60_likelihoods.sh


## Porter.

### Prepping dataset.

In [4]:
"""The Porter dataset provides us with the shared sequence of the 
fold-switching region of pairs of proteins. PDB IDs of the two proteins
are also provided. We can thus take a similar approach as OOD60:

1) Retrieve the corresponding chains from PDB structures and only 
consider pairs with large (>75%) continuous sequence overlap.

2) Crop the all-atom chain pair to the maximum sequence overlap region.

3) Use PDBFixer to fill in missing heavy atoms. 
"""

df = pd.read_excel(RAW_PORTER)

fold1s = df['Fold1'].tolist()
fold2s = df['Fold2'].tolist()
sequences = df['Sequence of fold-switching region'].tolist()

pdbl = PDBList()
parser = PDBParser(QUIET=True)

ppb = PPBuilder()
matched = False

for i, (id1, id2, seq) in tqdm(
    enumerate(zip(fold1s, fold2s, sequences), start=2),
    desc='Creating Porter dataset'
):
    pdb_id1 = id1[:4]
    pdb_id2 = id2[:4]

    chain_id1 = id1[-1]
    chain_id2 = id2[-1]

    # Retrieving original PDB files.
    pdb_file1 = pdbl.retrieve_pdb_file(pdb_id1, file_format='pdb', pdir='.', overwrite=True)
    structure1= parser.get_structure(pdb_id1, pdb_file1)
    pdb_file2 = pdbl.retrieve_pdb_file(pdb_id2, file_format='pdb', pdir='.', overwrite=True)
    structure2= parser.get_structure(pdb_id2, pdb_file2)
    os.remove(pdb_file1)
    if os.path.exists(pdb_file2): # Sometimes, it's two chains from the same PDB.
        os.remove(pdb_file2)

    chain1 = structure1[0][chain_id1]
    chain2 = structure2[0][chain_id2]

    # Extracting sequences from chains of interest. 
    try:
        residues1 = [res for res in chain1 if is_aa(res)]
        seq1 = ''.join(protein_letters_3to1[res.get_resname().capitalize()] for res in residues1)
        residues2 = [res for res in chain2 if is_aa(res)]
        seq2 = ''.join(protein_letters_3to1[res.get_resname().capitalize()] for res in residues2)
    except Exception as e:
        print('Likely a non-canonical amino acid.')
        print(f'Encountered error {e} for PDBs {id1, id2}')
        continue

    # Finding maximum common subsequence.
    seq1idxs, seq2idxs = longest_common_substring(seq1, seq2)
    
    cut_seq1 = seq1[seq1idxs[0]:seq1idxs[1]]
    cut_seq2 = seq2[seq2idxs[0]:seq2idxs[1]]
    assert cut_seq1 == cut_seq2, 'Mismatched sequences!'

    if seq not in cut_seq1:
        print('Sequence difference in fold-switching sequence. Skipping.')
        continue
    
    if max(len(seq1), len(seq2)) * 0.75 < len(cut_seq1):
        tempdir = tempfile.mkdtemp()
        outdir = os.path.join(FINAL_PORTER_DIR, str(i)) # Row number of the source excel sheet.
        outdir_amber = os.path.join(PORTER_AMBER, str(i))
        os.makedirs(outdir, exist_ok=True)
        os.makedirs(outdir_amber, exist_ok=True)
        
        crop1 = crop_chain_by_sequence(
            chain=chain1,
            matching_subseq=cut_seq1,
            output_pdb_path=os.path.join(tempdir, 'temp1.pdb')
        )
        crop2 = crop_chain_by_sequence(
            chain=chain2,
            matching_subseq=cut_seq1,
            output_pdb_path=os.path.join(tempdir, 'temp2.pdb')
        )
        if crop1 == -1 or crop2 == -1:
            print(f'One of PDBs {id1, id2} has different fold-switched sequences. Skipping.')
            continue

        '''Two prep steps occur: 1) using pdb4amber + tleap to create 
        the necessary Amber files for likelihood calculation 2) using 
        pdbfixer to create likelihood-ready structures. There are small
        differences and incompatibilities between pdb4amber and pdbfixer
        outputs, hence the necessity of using both.
        '''

        outname1 = os.path.join(outdir, f'{pdb_id1.upper()}_{chain_id1.upper()}.pdb') # eg. outdir/1XNT_A.pdb.
        outname2 = os.path.join(outdir, f'{pdb_id2.upper()}_{chain_id2.upper()}.pdb')

        # Using pdb4amber to create tleap inputs.
        pdb4amber_cmd = [
            "pdb4amber",
            "-i", f"{os.path.join(tempdir, 'temp1.pdb')}",
            "-o", f"{os.path.join(tempdir, 'temp1p.pdb')}",
            "--nohyd",
        ]
        result = subprocess.run(
            pdb4amber_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        pdb4amber_cmd = [
            "pdb4amber",
            "-i", f"{os.path.join(tempdir, 'temp2.pdb')}",
            "-o", f"{os.path.join(tempdir, 'temp2p.pdb')}",
            "--nohyd",
        ]
        result = subprocess.run(
            pdb4amber_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        # tleap-ing pdb4amber outputs.
        prmtop_savepath = os.path.join(outdir_amber, f'{pdb_id1.upper()}_{chain_id1.upper()}.prmtop')
        inpcrd_savepath = os.path.join(outdir_amber, f'{pdb_id1.upper()}_{chain_id1.upper()}.inpcrd')
        pdb_savepath = os.path.join(outdir_amber, f'{pdb_id1.upper()}_{chain_id1.upper()}.pdb')
        tleap_input = f"""
        source leaprc.ff99SBxildn
        mol = loadPDB "{os.path.join(tempdir, 'temp1p.pdb')}"
        saveAmberParm mol "{prmtop_savepath}" "{inpcrd_savepath}"
        savePDB mol "{pdb_savepath}"
        quit
        """
        result = subprocess.run(
            ["tleap", "-f", "-"],  # "-" means read from stdin
            input=tleap_input.encode(),
            check=True
        )

        # tleap-ing pdb4amber outputs.
        prmtop_savepath = os.path.join(outdir_amber, f'{pdb_id2.upper()}_{chain_id2.upper()}.prmtop')
        inpcrd_savepath = os.path.join(outdir_amber, f'{pdb_id2.upper()}_{chain_id2.upper()}.inpcrd')
        pdb_savepath = os.path.join(outdir_amber, f'{pdb_id2.upper()}_{chain_id2.upper()}.pdb')
        tleap_input = f"""
        source leaprc.ff99SBxildn
        mol = loadPDB "{os.path.join(tempdir, 'temp2p.pdb')}"
        saveAmberParm mol "{prmtop_savepath}" "{inpcrd_savepath}"
        savePDB mol "{pdb_savepath}
        quit
        """
        result = subprocess.run(
            ["tleap", "-f", "-"],  # "-" means read from stdin
            input=tleap_input.encode(),
            check=True
        )

        # Fixing saved PDBs of cropped all-atom chains and saving.
        fixer1 = PDBFixer(filename=os.path.join(tempdir, 'temp1.pdb'))
        fixer1.findMissingResidues()
        fixer1.findMissingAtoms()
        fixer1.addMissingAtoms()
        PDBFile.writeFile(fixer1.topology, fixer1.positions, open(outname1, 'w'))

        fixer2 = PDBFixer(filename=os.path.join(tempdir, 'temp2.pdb'))
        fixer2.findMissingResidues()
        fixer2.findMissingAtoms()
        fixer2.addMissingAtoms()
        PDBFile.writeFile(fixer2.topology, fixer2.positions, open(outname2, 'w'))

        write_yaml(cut_seq1, os.path.join(outdir, 'seq.yaml'))
        shutil.rmtree(tempdir)

Creating Porter dataset: 0it [00:00, ?it/s]

Creating Porter dataset: 1it [00:01,  1.57s/it]

Creating Porter dataset: 2it [00:02,  1.08s/it]

Sequence difference in fold-switching sequence. Skipping.


Creating Porter dataset: 3it [00:03,  1.07it/s]

...

Sequence difference in fold-switching sequence. Skipping.


Creating Porter dataset: 91it [02:12,  1.05it/s]

Sequence difference in fold-switching sequence. Skipping.


Creating Porter dataset: 92it [02:13,  1.45s/it]

Sequence difference in fold-switching sequence. Skipping.


In [7]:
DATA_DIR = '/home/ethanz/data_boltz_likelihood/porter'

job_script_path = "run_porter_likelihoods.sh"
exp_names = sorted(os.listdir(DATA_DIR))
head_inits = [
    os.path.join(DATA_DIR, exp_name)
    for exp_name in exp_names
]
yamls = [
    os.path.join(head_init, 'seq.yaml')
    for head_init in head_inits
]

# Define multiple argument sets for each run
runs = [
    {
        "exp_name": f"{exp_names[i]}_likelihoods",
        "head_init": head_inits[i],
        "yaml": yamls[i]
    }
    for i in range(len(exp_names))
]

with open(job_script_path, "w") as f:
    f.write("#!/bin/bash\n")
    f.write("#SBATCH --gres=gpu:1\n")
    f.write("#SBATCH --job-name=porter_likelihoods\n")

    f.write("source .venv/bin/activate\n\n")

    for run in runs:
        f.write(f'echo "Starting {run["exp_name"]}"\n')

        f.write("boltz predict \\\n")
        f.write(f"    {run["yaml"]} \\\n")
        f.write("    --atol 0.000001 \\\n")
        f.write("    --rtol 0.001 \\\n")
        f.write("    --model boltz1 \\\n")
        f.write("    --out_dir ./predictions/porter \\\n")
        f.write("    --use_msa_server \\\n")
        f.write("    --confidence False \\\n")
        f.write("    --output_format pdb \\\n")
        f.write("    --save_conditioning_args False \\\n")
        f.write("    --mode likelihood \\\n")
        f.write(f"    --experiment_name {run['exp_name']} \\\n")
        f.write(f"    --slurm_path {job_script_path} \\\n")
        f.write("    --likelihood_mode hutchinson \\\n")
        f.write("    --hutchinson_samples 20 \\\n")
        f.write(f'    --head_init {run["head_init"]} \\\n')

        f.write("\n")  # Separate commands with a blank line

print(f"Wrote job script to {job_script_path}")

Wrote job script to run_porter_likelihoods.sh
